# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os

## Load F1-score Results

In [4]:
# root_dir = "../experiments" # finetuning
# root_dir = "../experiments-not-finetuning" # pre-trained
root_dir = "../experiments-finetuning"
# root_dir = "../experiments-twitter" # 
# root_dir = "../experiments-swin"
data_paths = []

for dirs in os.listdir(root_dir):

    # if not dirs.split('-')[0] in ("openai", "deepseek"):
    if dirs.split('-')[0] == "deepseek" and dirs.split('-')[1] == "llama3":
        # print(dirs.split('-')[0])
        data_paths.append(os.path.join(root_dir, dirs+"/logs/test_logs.csv"))
        # print(dirs)
        # break

print(f"Amount of experiments: {len(data_paths)}")

Amount of experiments: 2


## Calculate the confidence interval

In [5]:
for data_path in sorted(data_paths):
    df = pd.read_csv(data_path)

    f1_scores = df["f1_score"].to_numpy()
    mean_f1 = np.mean(f1_scores)
    mean_acc = np.mean(df["accuracy"].to_numpy())
    # define the confidence level
    confidence_level = 0.95
    degrees_freedon = len(f1_scores)-1

    confidence_interval = stats.t.interval(
        confidence_level, 
        degrees_freedon, 
        loc=mean_f1, 
        scale=stats.sem(f1_scores)
    )

    if (data_path.split('/')[-3].split('.')[0][-1] in ("3", "5") and data_path.split('/')[-3].split('.')[0].split('-')[-2][-1] in ("3", "5")):
        print(f"\n\nProblem: {data_path.split('/')[-3].split('.')[0]}")

        # print(f"Max F1-score: {max(f1_scores)}")
        print(f"Average F1-score: {mean_f1*100:.2f}%")
        print(f"Time: {df['time'].to_numpy().sum()/3600:.2f} hours")
        # print(f"Average Acc: {mean_acc*100:.2f}%")
        # print(f"Median F1-score: {np.median(f1_scores)}")
        # print(f"Confidence interval 95%: {confidence_interval}")
        # print(f"Inteval: {abs(confidence_interval[0]-mean_f1)*100:.2f}%")
        print(f"Inteval: {df['f1_score'].to_numpy().std()*100:.2f}%")
        # print(f"Inteval: {abs(confidence_interval[0]-mean_acc)*100:.2f}%")
        confidence_interval = stats.t.interval(
            confidence_level, 
            degrees_freedon, 
            loc=max(f1_scores), 
            scale=stats.sem(f1_scores)
        )

        # print(f"Inteval: {abs(confidence_interval[0]-max(f1_scores))} - Interval: {abs(confidence_interval[1]-max(f1_scores))}")



Problem: deepseek-llama3-qlora-p3-alpha5
Average F1-score: 89.29%
Time: 6.67 hours
Inteval: 0.42%


Problem: deepseek-llama3-qlora-p5-alpha5
Average F1-score: 69.79%
Time: 2.97 hours
Inteval: 2.31%


## Evaluate Subjectivity Problem

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os
import io

In [2]:
root_dir = "../exps-tuning-handle-subjectivity"
data_paths = []

for dirs in os.listdir(root_dir):

    if dirs.split('-')[0] == "openai" and dirs.split('-')[1] == "modernbert":
        # print(dirs.split('-')[0])
        data_paths.append(os.path.join(root_dir, dirs+"/logs/test_logs.csv"))
        # print(dirs)
        # break

print(f"Amount of experiments: {len(data_paths)}")

Amount of experiments: 4


In [3]:
print(f"Amount of experiments: {len(data_paths)}")

# --- SECTION 3: EVALUATION LOOP ---
for data_path in sorted(data_paths):
    df = pd.read_csv(data_path)

    # 1. Extract Metrics
    pearsons = df["val_pearson"].to_numpy()
    mses = df["val_mse"].to_numpy()
    maes = df["val_mae"].to_numpy()
    
    # 2. Calculate Means
    mean_pearson = np.mean(pearsons)
    mean_mse = np.mean(mses)
    mean_mae = np.mean(maes)

    # 3. Calculate Time (Sum of seconds converted to hours)
    total_time_hours = df["time_sec"].sum() / 3600

    # 4. Statistics Configuration
    confidence_level = 0.95
    degrees_freedom = len(pearsons) - 1

    # 5. Calculate Confidence Interval (Focusing on Pearson as primary metric)
    ci_pearson = stats.t.interval(
        confidence_level, 
        degrees_freedom, 
        loc=mean_pearson, 
        scale=stats.sem(pearsons)
    )

    # --- PRINT RESULTS ---
    # Using generic label since directory parsing depends on your folder structure
    exp_name = data_path.split('/')[-1] if '/' in data_path else data_path
    
    print(f"\n--- Experiment: {data_path} ---")
    
    # Primary Metric: Pearson
    print(f"Average Pearson: {mean_pearson:.4f}")
    print(f"Pearson Std Dev: {np.std(pearsons, ddof=1):.4f}") # ddof=1 for sample std
    print(f"Pearson CI (95%): [{ci_pearson[0]:.4f}, {ci_pearson[1]:.4f}]")
    
    # Secondary Metrics: MSE / MAE
    print(f"Average MSE: {mean_mse:.4f} (Std: {np.std(mses, ddof=1):.4f})")
    print(f"Average MAE: {mean_mae:.4f} (Std: {np.std(maes, ddof=1):.4f})")
    
    # Time
    print(f"Total Time: {total_time_hours:.2f} hours")

Amount of experiments: 4

--- Experiment: ../exps-tuning-handle-subjectivity/openai-modernbert-experiment-p3-alpha3/logs/test_logs.csv ---
Average Pearson: 0.7817
Pearson Std Dev: 0.0137
Pearson CI (95%): [0.7647, 0.7987]
Average MSE: 0.3383 (Std: 0.0250)
Average MAE: 0.3336 (Std: 0.0259)
Total Time: 9.26 hours

--- Experiment: ../exps-tuning-handle-subjectivity/openai-modernbert-experiment-p3-alpha5/logs/test_logs.csv ---
Average Pearson: 0.9470
Pearson Std Dev: 0.0162
Pearson CI (95%): [0.9269, 0.9672]
Average MSE: 0.0980 (Std: 0.0299)
Average MAE: 0.1002 (Std: 0.0349)
Total Time: 5.22 hours

--- Experiment: ../exps-tuning-handle-subjectivity/openai-modernbert-experiment-p5-alpha3/logs/test_logs.csv ---
Average Pearson: 0.8344
Pearson Std Dev: 0.0280
Pearson CI (95%): [0.7996, 0.8692]
Average MSE: 0.6067 (Std: 0.1029)
Average MAE: 0.5700 (Std: 0.0550)
Total Time: 10.46 hours

--- Experiment: ../exps-tuning-handle-subjectivity/openai-modernbert-experiment-p5-alpha5/logs/test_logs.csv 